<a href="https://colab.research.google.com/github/shubham-senani/Marketing-Analytics/blob/main/MA5_Segmentation_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation Analytics

## SECTION 1 — DATA UNDERSTANDING & EXPLORATION

In [ ]:
# ===============================
# 1. IMPORT REQUIRED LIBRARIES
# ===============================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# ===============================
# 2. USER-CONTROLLED SETTINGS
# (Students ONLY need to change this section)
# ===============================


# ---- GOOGLE SHEETS CONFIGURATION ----
# Copy the Google Sheet ID from the URL
# Example URL:
# https://drive.google.com/file/d/1mQo6U4fJ_Nu-qm8gcIBw8YR7jih9lnfI/view?usp=sharing


GOOGLE_SHEET_ID = "1mQo6U4fJ_Nu-qm8gcIBw8YR7jih9lnfI"


# Construct downloadable CSV URL
DATA_URL = f"https://docs.google.com/uc?id={GOOGLE_SHEET_ID}&export=download"


# ---- SEGMENTATION VARIABLES (NUMERIC ONLY) ----
CLUSTER_VARS = [
"Annual Income (k$)",
"Spending Score (1-100)",
"Estimated Savings (k$)",
"Credit Score",
"Loyalty Years"
]


# ---- PROFILING VARIABLES (CATEGORICAL / DEMOGRAPHIC) ----
PROFILE_VARS = [
"Gender",
"Age Group",
"Preferred Category"
]


# ---- K-MEANS SETTINGS ----
K_RANGE = range(2, 9) # Range for elbow method
OPTIMAL_K = 4 # Final K chosen after elbow plot
RANDOM_STATE = 42 # Ensures reproducibility

In [ ]:
# ===============================
# 3. LOAD DATA FROM GOOGLE SHEETS
# ===============================


# Read CSV directly from Google Sheets
df = pd.read_csv(DATA_URL)


# Basic inspection
print("\nData Structure:\n")
print(df.info())
print("Data Preview:\n", df.head())


Data Structure:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              200 non-null    int64  
 1   Gender                  200 non-null    object 
 2   Age                     200 non-null    int64  
 3   Annual Income (k$)      200 non-null    int64  
 4   Spending Score (1-100)  200 non-null    int64  
 5   Age Group               196 non-null    object 
 6   Estimated Savings (k$)  200 non-null    float64
 7   Credit Score            200 non-null    int64  
 8   Loyalty Years           200 non-null    int64  
 9   Preferred Category      200 non-null    object 
dtypes: float64(1), int64(6), object(3)
memory usage: 15.8+ KB
None
Data Preview:
    CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)  \
0           1    Male   19                  15                      39   
1         

In [ ]:
# ===============================
# 4. CROSS-TAB ANALYSIS (BEFORE CLUSTERING)
# Purpose: Build intuition BEFORE applying algorithms
# ===============================


def cross_tab_analysis(data, var1, var2):
    """
    Creates a percentage-based cross-tabulation
    Useful for exploratory segmentation insight
    """
    ct = pd.crosstab(
    data[var1],
    data[var2],
    normalize="index"
    ) * 100

    print(f"\nCross-tab: {var1} vs {var2}")
    print(ct.round(2))

    return ct

In [ ]:
# Example exploratory cross-tabs
ct_age_pref = cross_tab_analysis(df, "Age Group", "Preferred Category")
ct_gender_pref = cross_tab_analysis(df, "Gender", "Preferred Category")


Cross-tab: Age Group vs Preferred Category
Preferred Category  Budget  Electronics  Fashion  Luxury
Age Group                                               
18-25                11.76         0.00    52.94   35.29
26-35                10.00         0.00    38.33   51.67
36-50                14.52        69.35     0.00   16.13
51-65                25.00        75.00     0.00    0.00
65+                   8.33        91.67     0.00    0.00

Cross-tab: Gender vs Preferred Category
Preferred Category  Budget  Electronics  Fashion  Luxury
Gender                                                  
Female               14.29        37.50    19.64   28.57
Male                 12.50        40.91    21.59   25.00


## SECTION 2 — SEGMENTATION MODEL (K-MEANS)

In [ ]:
# ===============================
# 5. DATA PREPARATION FOR K-MEANS
# ===============================


# Select numeric segmentation variables
X = df[CLUSTER_VARS]


# Standardization is CRITICAL for K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# ===============================
# 6. ELBOW METHOD (PLOTLY)
# Purpose: Determine optimal number of clusters
# ===============================


wss = []

# Create a KMeans model with k clusters

for k in K_RANGE:
    km = KMeans(
        n_clusters=k, # Number of clusters to form
        random_state=RANDOM_STATE, # Ensures reproducible results
        n_init=10 # Run KMeans 10 times with different centroid seeds
    )

    # Fit the KMeans model to the scaled feature data
    # This assigns each data point to the nearest cluster centroid

    km.fit(X_scaled)

    # Append the Within-Cluster Sum of Squares (WSS)
    # inertia_ measures how compact the clusters are
    # Lower inertia = tighter, better-defined clusters

    wss.append(km.inertia_)


elbow_df = pd.DataFrame({
    "Number of Clusters (K)": list(K_RANGE),
    "Within-Cluster Sum of Squares": wss
})


fig = px.line(
    elbow_df,
    x="Number of Clusters (K)",
    y="Within-Cluster Sum of Squares",
    markers=True,
    title="Elbow Method for Optimal Number of Clusters"
)


fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# ===============================
# 7. FINAL K-MEANS MODEL
# ===============================

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=RANDOM_STATE,
    n_init=25
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
df.head()

,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100),Age Group,Estimated Savings (k$),Credit Score,Loyalty Years,Preferred Category,Cluster
0,1,Male,19,15,39,18-25,11.10,456,3,Budget,1
1,2,Male,21,15,81,18-25,6.90,300,6,Luxury,1
2,3,Female,20,16,6,18-25,15.36,594,2,Budget,0
3,4,Female,23,16,77,18-25,7.79,300,6,Luxury,1
4,5,Female,31,17,40,26-35,12.47,480,5,Budget,1


In [ ]:
# ===============================
# 8. CLUSTER SIZE DISTRIBUTION
# ===============================

cluster_sizes = (
    df["Cluster"]
    .value_counts()
    .reset_index(name="Count")
    .rename(columns={"index": "Cluster"})
)

fig = px.bar(
    cluster_sizes,
    x="Cluster",
    y="Count",
    text="Count",
    title="Number of Customers in Each Cluster"
)

fig.update_layout(template="plotly_white")
fig.show()

## SECTION 3 — CLUSTER INTERPRETATION & PROFILING

In [ ]:
# ===============================
# 9. CLUSTER MEANS (SEGMENT DEFINITION)
# ===============================

cluster_means = df.groupby("Cluster")[CLUSTER_VARS].mean().round(2)
print("\nCluster Means (Segmentation Variables):")
print(cluster_means)


Cluster Means (Segmentation Variables):
         Annual Income (k$)  Spending Score (1-100)  Estimated Savings (k$)  \
Cluster                                                                       
0                     45.93                   38.40                   32.94   
1                     26.34                   71.41                   13.78   
2                     72.48                   67.03                   38.62   
3                     87.67                   17.72                   77.13   

         Credit Score  Loyalty Years  
Cluster                               
0              751.17           5.12  
1              416.38           5.93  
2              813.21           7.32  
3              850.00           4.39  


In [ ]:
# ===============================
# 10. CLUSTER VISUALIZATION (2D)
# ===============================

fig = px.scatter(
    df,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    color="Cluster",
    title="Customer Segments: Income vs Spending",
    hover_data=[
        "Age",
        "Estimated Savings (k$)",
        "Credit Score",
        "Loyalty Years",
        "Preferred Category"
    ]
)

fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# ===============================
# 11. PARALLEL COORDINATES (MULTI-VARIATE VIEW)
# ===============================

fig = px.parallel_coordinates(
    df,
    dimensions=CLUSTER_VARS,
    color="Cluster",
    color_continuous_scale=px.colors.diverging.Tealrose,
    title="Parallel Coordinates View of Customer Segments"
)

fig.show()

In [ ]:
# ===============================
# 12. CLUSTER PROFILING USING CROSS-TABS
# ===============================

def profile_by_cluster(data, profile_var):
    """
    Cross-tab profiling by cluster (column percentages)
    """
    ct = pd.crosstab(
        data[profile_var],
        data["Cluster"],
        normalize="columns"
    ) * 100

    print(f"\nProfile of {profile_var} by Cluster")
    print(ct.round(2))

    fig = px.imshow(
        ct,
        text_auto=".1f",
        title=f"{profile_var} Distribution by Cluster (%)",
        color_continuous_scale="Reds"
    )

    fig.update_layout(template="plotly_white")
    fig.show()


for var in PROFILE_VARS:
    profile_by_cluster(df, var)


Profile of Gender by Cluster
Cluster      0      1      2      3
Gender                             
Female   61.67  58.62  54.67  47.22
Male     38.33  41.38  45.33  52.78



Profile of Age Group by Cluster
Cluster        0      1      2      3
Age Group                            
18-25      22.81  53.57   0.00  16.67
26-35      21.05  42.86  40.00  16.67
36-50      38.60   3.57  28.00  50.00
51-65      15.79   0.00  17.33  16.67
65+         1.75   0.00  14.67   0.00



Profile of Preferred Category by Cluster
Cluster                 0      1      2      3
Preferred Category                            
Budget              31.67  27.59   0.00   0.00
Electronics         30.00   3.45  46.67  66.67
Fashion             38.33   3.45   6.67  33.33
Luxury               0.00  65.52  46.67   0.00
